In [ ]:
"""
Week 3 - Day 4
Comprehensive Analysis
=======================
Complete Week 3 analysis showing
PPO > DQN > Q-Learning > Baselines.

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS
)
from analysis.week3_analyzer import (
    train_all_agents,
    evaluate_all_agents,
    create_week3_dashboard,
    prove_ppo_superiority
)
from analysis.final_comparison import (
    create_final_summary_chart,
    save_final_results
)
from config import PPO, DQN

plt.style.use('seaborn-v0_8')
print("✅ Week 3 analysis modules loaded!")

In [ ]:
env = DynamicPricingEnv()

print("Training all agents with best configs...")
print("This takes about 5-8 minutes...\n")

rl_agents = train_all_agents(
    env, n_episodes=2000
)
print("\n✅ All agents trained!")

In [ ]:
print("Evaluating all agents (100 episodes)...")
results_df = evaluate_all_agents(
    env, rl_agents, n_eval=100
)

print("\n=== FINAL RANKINGS ===")
medals = ['🥇', '🥈', '🥉',
          '4️⃣', '5️⃣', '6️⃣', '7️⃣']
for i, row in results_df.iterrows():
    print(f"  {medals[i]} {row['Agent']:<15}: "
          f"${row['Mean Revenue']:.0f}")

In [ ]:
create_week3_dashboard(
    results_df, rl_agents,
    save_path='../results/week3_dashboard.png'
)

In [ ]:
proof = prove_ppo_superiority(
    env, rl_agents, n_test=200
)

In [ ]:
create_final_summary_chart(
    results_df,
    save_path='../results/final_comparison.png'
)

In [ ]:
final = save_final_results(
    results_df, proof
)

In [ ]:
# Final proof DQN learned behaviors
ppo_agent = rl_agents['PPO']
early_prices  = []
urgent_prices = []

for ep in range(100):
    state, _ = env.reset(seed=ep)
    done = False
    while not done:
        action = ppo_agent.select_action(
            state, training=False
        )
        price = PRICE_LEVELS[action]
        days  = int(state[1])
        if days >= 20:
            early_prices.append(price)
        elif days <= 5:
            urgent_prices.append(price)
        state, _, term, trunc, _ = (
            env.step(action)
        )
        done = term or trunc

avg_early  = np.mean(early_prices)
avg_urgent = np.mean(urgent_prices)
drop_pct   = (
    avg_early - avg_urgent
) / avg_early * 100

print("=== PPO BEHAVIOR PROOF ===\n")
print(f"  Early price  : ${avg_early:.0f}")
print(f"  Urgent price : ${avg_urgent:.0f}")
print(f"  Price drop   : -{drop_pct:.1f}%")
if avg_urgent < avg_early:
    print(f"\n  ✅ PPO DROPS PRICES near deadline!")

In [ ]:
best       = results_df.iloc[0]
best_bl    = results_df[
    ~results_df['Agent'].isin([
        'PPO', 'DQN', 'Q-Learning'
    ])
]['Mean Revenue'].max()
improvement = (
    best['Mean Revenue'] - best_bl
) / best_bl * 100

print("╔══════════════════════════════════════════╗")
print("║    WEEK 3 DAY 4 — ANALYSIS COMPLETE!    ║")
print("╠══════════════════════════════════════════╣")
print("║  FINAL RANKINGS:                         ║")
for i, row in results_df.iterrows():
    print(f"║  {medals[i]} {row['Agent']:<18}: "
          f"${row['Mean Revenue']:<8.0f}     ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Best Agent    : {best['Agent']:<24} ║")
print(f"║  Best Revenue  : "
      f"${best['Mean Revenue']:.0f}"
      f"{'':<22} ║")
print(f"║  vs Baseline   : {improvement:+.1f}%"
      f"{'':<23} ║")
print("╠══════════════════════════════════════════╣")
print("║  STATISTICAL PROOF:                      ║")
for name, p in proof.items():
    sig = '✅' if p['significant'] else '⚠️'
    print(f"║  {sig} PPO vs {name:<10}: "
          f"p={p['p_value']:.4f}"
          f"{'':<11} ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Week 3 Wrap Up 📝            ║")
print("╚══════════════════════════════════════════╝")